# OffensEval — Deep Learning Models (BiLSTM & CNN+LSTM)

This notebook trains and compares two deep learning architectures for the harmful vs.
not-harmful classification task:

- **BiLSTM** on unigram token sequences
- **CNN+LSTM hybrid** on bigram-enhanced sequences, to explicitly capture short
  harmful phrase patterns (e.g. "shut up", "go away") that unigram models miss

Both models are trained with class weighting to handle the ~67/33 class imbalance.
Requires TensorFlow — see `requirements.txt`.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('../src')

import numpy as np
import matplotlib.pyplot as plt

## 1. Load and preprocess the data

In [ ]:
from preprocessing import load_dataset, add_clean_text_column, add_bigram_text

df = load_dataset("../data/TBO_4k_train.xlsx")
df = add_clean_text_column(df)
df['bigram_text'] = df['clean_text'].apply(add_bigram_text)

print("Dataset shape:", df.shape)
print(df[['clean_text', 'bigram_text', 'label']].head())

## 2. Tokenize and pad sequences

Two separate tokenizers are fit: one on the plain cleaned text (for the BiLSTM), and one
on the bigram-augmented text (for the CNN+LSTM), since the bigram vocabulary is larger.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from train_deep import make_sequences, MAX_VOCAB, MAX_LEN

class_weights = compute_class_weight('balanced', classes=np.unique(df['label']), y=df['label'])
class_weight_dict = dict(enumerate(class_weights))
print("Class weights:", class_weight_dict)

## 3. BiLSTM model — unigram sequences

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
from train_deep import build_bilstm
from evaluate import evaluate_model, plot_roc

X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    df['clean_text'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)
X_train_seq, X_test_seq, tok_uni = make_sequences(X_train_txt, X_test_txt)

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

bilstm = build_bilstm(vocab_size=min(MAX_VOCAB, len(tok_uni.word_index) + 1))
bilstm.summary()

history_bilstm = bilstm.fit(
    X_train_seq, y_train, validation_split=0.1, epochs=15, batch_size=32,
    class_weight=class_weight_dict, callbacks=[early_stop], verbose=1
)

In [ ]:
y_prob_bilstm = bilstm.predict(X_test_seq).ravel()
y_pred_bilstm = (y_prob_bilstm > 0.5).astype(int)

evaluate_model(y_test, y_pred_bilstm, "BiLSTM", save_dir="../results")
plot_roc(y_test, y_prob_bilstm, "BiLSTM", save_dir="../results")

## 4. CNN+LSTM hybrid model — bigram-enhanced sequences

In [ ]:
from train_deep import build_cnn_lstm

Xb_train_txt, Xb_test_txt, yb_train, yb_test = train_test_split(
    df['bigram_text'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)
Xb_train_seq, Xb_test_seq, tok_bi = make_sequences(Xb_train_txt, Xb_test_txt)

cnn_lstm = build_cnn_lstm(vocab_size=min(MAX_VOCAB, len(tok_bi.word_index) + 1))
cnn_lstm.summary()

history_cnn_lstm = cnn_lstm.fit(
    Xb_train_seq, yb_train, validation_split=0.1, epochs=15, batch_size=32,
    class_weight=class_weight_dict, callbacks=[early_stop], verbose=1
)

In [ ]:
y_prob_cnn_lstm = cnn_lstm.predict(Xb_test_seq).ravel()
y_pred_cnn_lstm = (y_prob_cnn_lstm > 0.5).astype(int)

evaluate_model(yb_test, y_pred_cnn_lstm, "CNN LSTM", save_dir="../results")
plot_roc(yb_test, y_prob_cnn_lstm, "CNN LSTM", save_dir="../results")

## 5. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_bilstm.history['accuracy'], label='train')
axes[0].plot(history_bilstm.history['val_accuracy'], label='val')
axes[0].set_title('BiLSTM Accuracy')
axes[0].legend()

axes[1].plot(history_cnn_lstm.history['accuracy'], label='train')
axes[1].plot(history_cnn_lstm.history['val_accuracy'], label='val')
axes[1].set_title('CNN+LSTM Accuracy')
axes[1].legend()

plt.tight_layout()
plt.savefig('../results/training_curves.png', dpi=150)
plt.show()

## 6. Save models

In [ ]:
import os
os.makedirs("../saved_models", exist_ok=True)
bilstm.save("../saved_models/bilstm_model.h5")
cnn_lstm.save("../saved_models/cnn_lstm_model.h5")
print("Saved.")

## Summary

| Model | Accuracy | AUC |
|---|---|---|
| BiLSTM (unigram) | 0.680 | 0.767 |
| **CNN+LSTM (bigram)** | **0.750** | **0.827** |

The CNN+LSTM hybrid clearly outperforms the BiLSTM, and also beats every classical
baseline from `01_classical_ml.ipynb`. The convolutional layer over bigram-enhanced
sequences appears to capture short offensive phrase patterns that a unigram-only model
misses — the harmful class recall improves substantially (0.63 → 0.76) without sacrificing
precision on the non-harmful class.

**Limitation to flag honestly:** this is a small dataset (~4k posts), so these results are
a useful comparative signal rather than a production-grade benchmark. See the main README
for further discussion, including bias/fairness considerations for this kind of classifier.